# Test Prob-RVEA (2022): single problem plot

Colab-ready full notebook. It runs one selected problem/seed and shows the seed-1 objective-space plot with `f_sur`, `f_real`, and `HV_xy_shift`.


### Package


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
import types

sys.dont_write_bytecode = True

if "imp" not in sys.modules and importlib.util.find_spec("imp") is None:
    # pyDOE2 1.3.0 imports the removed module but does not use it for LHS.
    sys.modules["imp"] = types.ModuleType("imp")

DEPENDENCIES = {
    "pymoo": "pymoo==0.6.1.6",
    "pyDOE2": "pyDOE2",
    "GPy": "GPy",
    "yaml": "pyyaml",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "diversipy": "diversipy",
    "optproblems": "optproblems",
    "graphviz": "graphviz",
    "matplotlib": "matplotlib",
    "plotly": "plotly"
}

for import_name, pip_name in DEPENDENCIES.items():
    try:
        importlib.import_module(import_name)
        print(f"{import_name} is available.")
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pip_name])


try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')

from pathlib import Path

code_path = Path("/content/drive/MyDrive/2026 Indicator_misleading/src")
for repo_root in (Path.cwd().resolve(), Path.cwd().resolve().parent, code_path.parent, code_path):
    if (repo_root / "baseline" / "batch_experiments.py").exists():
        repo_root_string = str(repo_root)
        while repo_root_string in sys.path:
            sys.path.remove(repo_root_string)
        sys.path.insert(0, repo_root_string)
        print(f"Using repository root: {repo_root}")
        break
else:
    raise FileNotFoundError("Could not locate baseline/batch_experiments.py")

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)
sys.modules.pop("baseline.batch_experiments", None)
sys.modules.pop("baseline", None)
import baseline.batch_experiments as batch_experiments

print(f"Loaded baseline runner: {batch_experiments.__file__}")
run_prob_rvea_suite = batch_experiments.run_prob_rvea_suite
print_gap_improvement_table = batch_experiments.print_gap_improvement_table



### Single test run


In [ ]:
# ===== Test controls: run one selected problem, this baseline method, one seed, and draw xy-shift plot =====
SELECTED_PROBLEM = "dtlz1"
SELECTED_SEED = 1
METHOD_NAME = "Prob-RVEA"

import os
import tempfile
from pathlib import Path
import yaml

os.environ.pop("DISABLE_HV_PLOTS", None)
os.environ["HV_TEST_PLAIN_PLOT"] = "1"
os.environ["HV_TEST_SAVE_SVG"] = "1"
os.environ["HV_TEST_SVG_PREFIX"] = "negative1"
os.environ["HV_TEST_SEPARATE_LEGEND"] = "1"
os.environ["HV_TEST_SVG_DIR"] = str(Path("experiments/plot_sur_real/svg").resolve())

base_config_path = Path(getattr(batch_experiments, "DEFAULT_CONFIG_PATH"))
with open(base_config_path, "r", encoding="utf-8") as config_file:
    single_config = yaml.safe_load(config_file)

single_config["problem_names"] = [SELECTED_PROBLEM]
single_config["seed_start"] = int(SELECTED_SEED)
single_config["seed_end"] = int(SELECTED_SEED) + 1

tmp_config_path = Path(tempfile.gettempdir()) / f"test_exp_single_{SELECTED_PROBLEM}_seed{SELECTED_SEED}.yaml"
with open(tmp_config_path, "w", encoding="utf-8") as config_file:
    yaml.safe_dump(single_config, config_file, sort_keys=False)

print(f"Test run | problem={SELECTED_PROBLEM} | method={METHOD_NAME} | seed={SELECTED_SEED} | config={tmp_config_path}")
all_results = run_prob_rvea_suite(config_path=tmp_config_path)


### Gap improvement table


In [ ]:
# Aggregate gap-improvement table printing is skipped for this single plotting test.
gap_improvement_table = None
